In [0]:
from pyspark.sql.functions import *
from pyspark.dbutils import *
from datetime import datetime, date

In [0]:
## variables

database = "ipl_database"
bronze = "bronze"
silver = "silver"

Creating a table that will has all the match ids, venue, winner, toss winner, wheather a match was day/night, match date, teams etc.

In [0]:
%sql
select id,batsman from ipl_database.bronze.batsman_raw
group by 1,2
having count(*) > 1
order by id

In [0]:
%skip
df = spark.readStream.table("ipl_database.bronze.batsman_data_raw") \
    .select(
        "id",
        "batting_team",
        "bowling_team",
        "date",
        "venue",
        "winner",
        "toss",
        "file_name"
    )

In [0]:
## get todays dates to process only new data

today_date = date.today()
print(today_date)

In [0]:
## batting raw data

if spark.catalog.tableExists(f"{database}.{silver}.match_details"):
    df = spark.table(f"{database}.{bronze}.batsman_raw") \
        .filter(
            to_date(col("updated_at")) >= today_date
        ) \
        .select(
            "id",
            "batting_team",
            "bowling_team",
            "date",
            "venue",
            "winner",
            "toss"
        ) 
    ## get max match_id for the particular season
    md_silver = spark.table(f"{database}.{silver}.match_details")
    max_id = md_silver.groupBy("season").agg(
        max("match_number").alias("max_match_id")
    ).distinct()
    print(max_id.display())
else:
    df = spark.table(f"{database}.{bronze}.batsman_raw") \
        .select(
            "id",
            "batting_team",
            "bowling_team",
            "date",
            "venue",
            "winner",
            "toss"
        ) 
    max_id = spark.createDataFrame([("2008", 1)], ["season", "max_match_id"])
    # print(max_id.display())

In [0]:
%skip
df.display(10)

## 00 Drop duplicates

In [0]:
df_dedup = df.dropDuplicates(['id'])

## 01 Assigining UUID to match_id

In [0]:
df_uuid = (
  df_dedup.withColumn(
    "match_UUID",
    substring(sha2(concat_ws("-","id","batting_team","bowling_team","date"), 256),1,36)
  )
)

In [0]:
%skip
from pyspark.sql.functions import sha2, concat_ws
df_test = df_dedup.withColumn(
    "test_uuid",
    substring(sha2(concat_ws("-","id","batting_team","bowling_team","date"), 256),1,36)
)

In [0]:
%skip
df_test.display(10)

In [0]:
%skip
df_uuid.display(10)

In [0]:
## Check dist uuid are generated for each match id

df_uuid.groupBy("match_UUID").agg(
    count("*").alias("counts")
).filter(col("counts") > 1).display()

## 02 is_day 
 - 1 day match
 - 0 night match

In [0]:
df2 = df_uuid.withColumn(
  "is_day",
  when(regexp_extract("id",r'\((.*?)\)',1) == 'D/N',1).otherwise(0)
)

In [0]:
%skip
df2.display()

## 03 toss winner

In [0]:
df3 = df2.withColumn(
  "toss_winner",
  trim(split(col("toss"),",")[0])
) \
.withColumn(
  "toss_decision",
  trim(split(col("toss"),",")[1])
) 
# df3.display()

## 04 match winner

In [0]:
teams_abbrivations = [
  {
    "team_name": "Chennai Super Kings",
    "short_name": "CSK"
  },
  {
    "team_name": "Mumbai Indians",
    "short_name": "MI"
  },
  {
    "team_name": "Royal Challengers Bangalore",
    "short_name": "RCB"
  },
  {
    "team_name": "Kolkata Knight Riders",
    "short_name": "KKR"
  },
  {
    "team_name": "Delhi Capitals",
    "short_name": "DC"
  },
  {
    "team_name": "Punjab Kings",
    "short_name": "PBKS"
  },
  {
    "team_name": "Rajasthan Royals",
    "short_name": "RR"
  },
  {
    "team_name": "Sunrisers Hyderabad",
    "short_name": "SRH"
  },
  {
    "team_name": "Lucknow Super Giants",
    "short_name": "LSG"
  },
  {
    "team_name": "Gujarat Titans",
    "short_name": "GT"
  }
]

In [0]:
match_abbrivation_df = spark.createDataFrame(teams_abbrivations)
# match_abbrivation_df.display()

In [0]:
from pyspark.sql import Window
window_spec = Window.orderBy(col("match_date"),col("id"))

df4 = df3.withColumn(
    "match_winner",
    trim(split(col("winner")," ")[0])
) \
.withColumn(
    "match_date",
    to_date(trim(col("date")), "MMMM dd yyyy")
).alias("a") \
.join(
    match_abbrivation_df.alias("b"),
    on = col("a.match_winner") == col("b.short_name"),
    how="left"
) \
.withColumn(
    "match_rank",
    rank().over(window_spec)
) \
.orderBy(col("match_date"),col("id"))
# df4.display()

In [0]:
%skip
df4.display()

## 05 resolve date and venue and rename columns names

In [0]:
df6 = (
  df4
  .withColumn(
    "venue",
    trim(col("venue"))
  )
  .withColumn(
    "team_1",
    trim(col("batting_team"))
  )
  .withColumn(
    "team_2",
    trim(col("bowling_team"))
  )
  .withColumn(
    "season",
    year(col("match_date"))
  )
  .join(
    max_id,
    on="season",
    how="left"
  )
  .withColumn(
    "match_number",
    col("match_rank") + coalesce(col("max_match_id"),lit(0))
  )
)
# df6.display()

## Final table

In [0]:
df_fin = df6.select(
    "match_UUID",
    "match_number",
    "match_date",
    "id",
    "team_1",
    "team_2",
    "venue",
    "toss_winner",
    "toss_decision",
    col("team_name").alias("match_winner"),
    "is_day",
    "season"
).orderBy(col("id"))

In [0]:

df_fin.display()

## Saving delta table

In [0]:
%skip
## overwrite or append

from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark,file_location):
    df_fin.writeStream \
        .format("delta") \
        .option("checkpointLocation","abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/checkpoints/") \
        .mode("overwrite") \
        .option("mergeschema","true") \
        .trigger(availableNow=True) \
        .start("abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/")
    spark.sql(f'''
              create table ipl_database.silver.match_details
              using delta
              location 'abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/'
              ''')
else:
    ## upsert ops
    df = spark.table(f"ipl_database.silver.match_details")
    df_fin.writeStream \
        .format("delta") \
        .option("checkpointLocation","abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/checkpoints/") \
        .option("mergeSchema","true") \
        .merge(
            source = df_fin.alias("updates"),
            target = df.alias("match_details")
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .trigger(availableNow=True) \
        .start("abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/")

In [0]:
%skip
## processed files 
try:
    new_data = df_fin.select("match_number","season").distinct()
    new_data.createOrReplaceTempView("new_data")
    spark.sql(f'''
              merge into ipl_database.silver.match_control mc
              using new_data nd
              on mc.match_number = nd.match_number and mc.season = nd.season
              when matched then update set *
              when not matched then insert *
              ''')
    print("append sucessfull.")
except:
    processed_files = df_fin.select("match_number","season").distinct()
    processed_files.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/control_table/")
    spark.sql(f'''
              create table ipl_database.silver.match_control
              using delta
              location "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/control_table/"
              ''')
    print("write sucessfull.")

In [0]:
%skip
df_fin.display()

In [0]:
%skip
from delta.tables import DeltaTable

delta_tables = DeltaTable.forPath(spark, "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/")

history_df = delta_tables.history()
history_df.display()

In [0]:
# delta_tables.restoreToVersion(3)

In [0]:
# spark.sql('refresh table ipl_database.silver.match_details')

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.match_details"):
        df_fin.createOrReplaceTempView("df_fin")
        spark.sql(f'''
                merge into ipl_database.silver.match_details mc
                using df_fin nd
                on mc.season = nd.season and mc.id = nd.id
                -- when matched then update set *
                when not matched then insert *
                ''')
        # Upsert in not ideal here, switching to append process
        print("append sucessfull.")
    else:
        df_fin.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("ipl_database.silver.match_details")
        # spark.sql(f'''
        #         create table ipl_database.silver.match_details
        #         using delta
        #         location "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/"
        #         ''')
        print("write sucessfull.")
except Exception as e:
    print(f"Error while write/append,{e}")

In [0]:
%skip
%sql
select * from ipl_database.silver.match_details

In [0]:
%skip
## write final match details table

from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/"):
    df_fin.write.mode("overwrite") \
    .format("delta") \
    .save("abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/") 


    spark.sql(f'''
                create table ipl_database.silver.match_details
                using delta
                location "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/"
                ''')
    print("write sucess!")
else:
    df_fin.write.format("delta") \
        .mode("append") \
        .save("abfss://silver@ipldatastorageaccount.dfs.core.windows.net/match_details/delta/")
    print("append sucess")